# Análise Estatística
```
Tema: Impacto da IA e das redes sociais na saúde e no desempenho dos estudantes
```
---

In [ ]:
import time
import polars as pl
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

ARQUIVO = "estudantes.parquet"

# >>> Se a detecção automática errar, preencha manualmente (ex.: "horas_uso_ia")
COL_IA = None
COL_NOTA = None

inicial = pl.scan_parquet(ARQUIVO)
esquema = inicial.collect_schema()
colunas = esquema.names()
numericas = [c for c, t in esquema.items() if t.is_numeric()]

def candidatas(tokens, entre):
    return [c for c in entre if set(c.split("_")) & tokens]

cand_ia = candidatas({"ia", "ai", "chatgpt", "llm"}, numericas)
cand_nota = candidatas({"nota", "notas", "gpa", "desempenho", "rendimento"}, numericas)

COL_IA = COL_IA or next((c for c in cand_ia if "horas" in c), cand_ia[0] if cand_ia else None)
COL_NOTA = COL_NOTA or (cand_nota[0] if cand_nota else None)

print("Colunas com 'ia' no nome (qualquer tipo):", candidatas({"ia", "ai", "chatgpt", "llm"}, colunas))
print("Coluna de IA em uso   :", COL_IA)
print("Coluna de nota em uso :", COL_NOTA)
if COL_IA is None:
    print("ATENÇÃO: nenhuma coluna numérica de IA encontrada. Veja a lista 'colunas' e preencha COL_IA.")

# Métricas de saúde e desempenho usadas nos resumos
METRICAS = list(dict.fromkeys(
    c for c in ["horas_redes_sociais", "horas_sono", "pontuacao_vicio_digital", "indice_bem_estar",
                "taxa_frequencia_aulas", "capacidade_atencao_minutos", "pontuacao_risco_academico",
                COL_IA, COL_NOTA]
    if c and c in colunas
))

def perfil(lf, grupo):
    """Resumo por grupo: nº de estudantes + média das métricas do tema."""
    return (
        lf.group_by(grupo)
        .agg([pl.len().alias("estudantes")] +
             [pl.col(c).mean().round(2).alias(f"media_{c}") for c in METRICAS])
        .sort(grupo)
    )

## Big Data e Processamento Otimizado com Python II

### Manipulação do Parquet
---

In [ ]:
display(inicial.collect())
print(f"{len(colunas)} colunas")

### Filtros
```
Para selecionar linhas baseadas em condições lógicas (semelhante ao WHERE do SQL).
```
---

In [ ]:
# Filtro composto (E): muito tempo em redes sociais e pouco sono
q_risco = inicial.filter(
    (pl.col("horas_redes_sociais") > 5) & (pl.col("horas_sono") <= 6)
)

# Filtro com lista (IS IN): estudantes de países desenvolvidos x demais
q_desenvolvidos = inicial.filter(pl.col("nivel_desenvolvimento").is_in(["Desenvolvido"]))
q_demais = inicial.filter(pl.col("nivel_desenvolvimento").is_in(["Em desenvolvimento", "Subdesenvolvido"]))

total = inicial.select(pl.len()).collect().item()
n_risco = q_risco.select(pl.len()).collect().item()
print(f"Alto uso de redes (>5h) e pouco sono (<=6h): {n_risco:,} estudantes ({n_risco/total:.1%})")
print(f"Países desenvolvidos: {q_desenvolvidos.select(pl.len()).collect().item():,}")
print(f"Demais países: {q_demais.select(pl.len()).collect().item():,}")

display(q_risco.head(5).collect())

# Filtro por uso intenso de IA (acima do percentil 75)
if COL_IA:
    q_ia_alto = inicial.filter(pl.col(COL_IA) > pl.col(COL_IA).quantile(0.75))
    print(f"Uso intenso de IA (> P75 de {COL_IA}): {q_ia_alto.select(pl.len()).collect().item():,} estudantes")
    display(q_ia_alto.head(5).collect())

### Seleção e Exclusão
```
Para escolher ou remover colunas específicas.
```
---

In [ ]:
# Selecionar apenas as colunas do tema (redes, IA, saúde e desempenho)
cols_tema = ["id_estudante", "pais", "idade", "nivel_escolaridade"] + METRICAS
q_tema = inicial.select(list(dict.fromkeys(cols_tema)))

# .drop() remove as colunas listadas
q_sem_infra = inicial.drop(["indice_infraestrutura_internet", "velocidade_media_internet_mbps"])

display(q_tema.head(5).collect())
print("q_tema:", q_tema.collect_schema().len(), "colunas | q_sem_infra:", q_sem_infra.collect_schema().len(), "colunas")

### Condicionais
```
Criar novas colunas com base em lógica (semelhante ao IF do Excel ou CASE WHEN do SQL).
```
---

In [ ]:
# Categorias de sono, uso de redes sociais e uso de IA (tercis)
exprs = [
    pl.when(pl.col("horas_sono") <= 6).then(pl.lit("1-Baixo (≤6h)"))
      .when(pl.col("horas_sono") < 8).then(pl.lit("2-Médio (6-8h)"))
      .otherwise(pl.lit("3-Bom (≥8h)")).alias("categoria_sono"),

    pl.when(pl.col("horas_redes_sociais") < 2).then(pl.lit("1-Baixo (<2h)"))
      .when(pl.col("horas_redes_sociais") < 5).then(pl.lit("2-Moderado (2-5h)"))
      .otherwise(pl.lit("3-Alto (≥5h)")).alias("categoria_redes"),
]
if COL_IA:
    exprs.append(
        pl.when(pl.col(COL_IA) <= pl.col(COL_IA).quantile(0.33)).then(pl.lit("1-Baixo uso de IA"))
          .when(pl.col(COL_IA) <= pl.col(COL_IA).quantile(0.66)).then(pl.lit("2-Médio uso de IA"))
          .otherwise(pl.lit("3-Alto uso de IA")).alias("categoria_ia")
    )

q_cat = inicial.with_columns(exprs)
cols_cat = [c for c in ["horas_sono", "categoria_sono", "horas_redes_sociais", "categoria_redes", COL_IA, "categoria_ia"] if c]
display(q_cat.select(cols_cat).head(10).collect())

### Agrupamento (group_by + agg)
```
Compara a saúde e o desempenho médios entre os grupos de uso.
```
---

In [ ]:
# Saúde e desempenho médios por nível de uso de redes sociais
display(perfil(q_cat, "categoria_redes").collect())

In [ ]:
# ... por nível de sono
display(perfil(q_cat, "categoria_sono").collect())

In [ ]:
# ... por nível de uso de IA
if COL_IA:
    display(perfil(q_cat, "categoria_ia").collect())

In [ ]:
# Cruzamento: redes sociais x sono -> bem-estar e frequência
cruz = (
    q_cat.group_by(["categoria_redes", "categoria_sono"])
    .agg(
        pl.len().alias("estudantes"),
        pl.col("indice_bem_estar").mean().round(2).alias("media_bem_estar"),
        pl.col("taxa_frequencia_aulas").mean().round(2).alias("media_frequencia"),
    )
    .sort(["categoria_redes", "categoria_sono"])
)
display(cruz.collect())

In [ ]:
# Por nível de escolaridade (ordenado por MÉDIA, não por soma)
display(perfil(inicial, "nivel_escolaridade").sort("media_horas_redes_sociais", descending=True).collect())

In [ ]:
# Top 10 países com maior pontuação média de vício digital
display(
    perfil(inicial, "pais")
    .sort("media_pontuacao_vicio_digital", descending=True)
    .head(10)
    .collect()
)

### Fatiamento
```
No modo Lazy, o slice é eficiente: o Polars lê apenas o necessário do disco.
```
---

In [ ]:
q_head = inicial.head(10)          # primeiras 10 linhas
q_meio = inicial.slice(100, 50)    # offset 100, 50 linhas

display(q_head.collect())
display(q_meio.collect())

In [ ]:
# Recorte de linhas específicas (linhas 0, 10 e 50)
q_recorte = (
    inicial
    .with_row_index("id_linha")
    .filter(pl.col("id_linha").is_in([0, 10, 50]))
    .select(["pais", "nivel_desenvolvimento", "horas_redes_sociais", "horas_sono"])
)
display(q_recorte.collect())

### Otimização: texto (`String`) x `Categorical`
```
Colunas de texto repetido (país, nível de desenvolvimento) ocupam menos memória e agrupam mais rápido como Categorical.
O pl.StringCache() está obsoleto (substituído por pl.Categories); aqui comparamos String x Categorical diretamente.
```
---

In [ ]:
def teste(usar_categorical, repeticoes=3):
    tempos = []
    for _ in range(repeticoes):
        inicio = time.perf_counter()
        q = pl.scan_parquet(ARQUIVO)
        if usar_categorical:
            q = q.with_columns(
                pl.col("pais").cast(pl.Categorical),
                pl.col("nivel_desenvolvimento").cast(pl.Categorical),
            )
        resultado = (
            q.group_by(["pais", "nivel_desenvolvimento"])
            .agg(
                pl.col("pontuacao_vicio_digital").mean().alias("media"),
                pl.col("pontuacao_vicio_digital").std().alias("desvio"),
            )
            .sort("media", descending=True)
            .collect()          # collect() DENTRO da medição
        )
        tempos.append(time.perf_counter() - inicio)
    return min(tempos)

t_str = teste(False)
t_cat = teste(True)

col_str = pl.read_parquet(ARQUIVO, columns=["pais"])["pais"]
col_cat = col_str.cast(pl.Categorical)

print(f"String      -> {t_str:.4f}s | memória da coluna 'pais': {col_str.estimated_size('mb'):.2f} MB")
print(f"Categorical -> {t_cat:.4f}s | memória da coluna 'pais': {col_cat.estimated_size('mb'):.2f} MB")